# Fase 2 — Notebook 3: validación técnica y resultados

**Proyecto:** Sobreduración en la titulación de la educación superior chilena (2020–2025)
**Equipo:** «Integrante 1», «Integrante 2», «Integrante 3»  ·  **Grupo:** «N»

---

**Objetivo de este notebook (OE6).** Verificar técnicamente el conjunto
analítico producido en el notebook anterior y presentar los resultados
exploratorios que responden a las preguntas planteadas en la Fase 1.

La verificación se realiza en tres niveles: **reglas de validación** sobre el
conjunto completo, **pruebas automatizadas** sobre las funciones del pipeline y
**comprobación de reproducibilidad** re-ejecutando el flujo de extremo a extremo.

## 1. Entorno y carga del conjunto analítico

In [ ]:
# Celda de arranque: hace importable el paquete `src` sin instalar el proyecto
# y funciona igual si el notebook se abre desde la raiz o desde su subcarpeta.
import sys
from pathlib import Path

RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

from src import config

print("Raiz del proyecto:", RAIZ)
for clave, valor in config.describir_entorno().items():
    print(f"  {clave:12s} {valor}")

In [ ]:
from src import pipeline, transformacion as tr, validacion, viz

viz.aplicar_estilo()

df = pipeline.cargar_analitico()
print(f"Conjunto analítico: {len(df):,} registros x {df.shape[1]} columnas")
print(f"Cobertura: {int(df['cat_periodo'].min())}-{int(df['cat_periodo'].max())}")

## 2. Validación mediante motor de reglas

La verificación no se deja a inspecciones visuales: está implementada como un
motor de reglas en `src/validacion.py`. Cada regla es un objeto con nombre,
descripción, severidad y una función de comprobación; el validador las ejecuta
todas y entrega una tabla de veredictos.

Las reglas **críticas** invalidan el conjunto si fallan; las **advertencias**
señalan comportamientos que deben documentarse pero no impiden continuar.

In [ ]:
validador = validacion.validador_estandar()
reporte = validador.ejecutar(df)

print(reporte[["regla", "severidad", "estado", "detalle"]].to_string(index=False))
print("\n" + validador.resumen())

reporte.to_csv(config.TABLES_DIR / "reporte_validacion.csv", index=False)
assert validador.aprobado, "El conjunto no supera las reglas críticas"

### 2.1 Análisis de la advertencia detectada

La única regla no aprobada es de severidad *advertencia*: en cerca de un 18 % de
los registros el año del proceso (`cat_periodo`) no coincide con el año de la
fecha del título. La comprobación siguiente muestra que se trata de un desfase
administrativo conocido y no de un error de procesamiento: son títulos obtenidos
al cierre de un año que el sistema informa en el proceso siguiente, o al revés.

In [ ]:
desfase = (df["cat_periodo"].astype(int) - df["anio_titulo"].astype(int))
resumen_desfase = desfase.value_counts().sort_index()
resumen_desfase = pd.DataFrame({
    "desfase_anios": resumen_desfase.index,
    "registros": resumen_desfase.to_numpy(),
    "porcentaje": (100 * resumen_desfase / len(df)).round(2).to_numpy(),
})
print(resumen_desfase.to_string(index=False))

meses_desfasados = df.loc[desfase != 0, "mes_titulo"].value_counts().sort_index()
print("\nMes de titulación de los registros desfasados:")
print(meses_desfasados.to_string())
print("\nEl desfase se concentra en los meses extremos del año, lo que confirma")
print("que corresponde al calendario administrativo del proceso y no a un error.")

## 3. Pruebas automatizadas del pipeline

La suite cubre casos normales, casos límite (bordes de cada regla) y
excepciones (entradas inválidas que deben fallar de forma controlada). Se
ejecuta desde el notebook para dejar la evidencia dentro del documento.

In [ ]:
import subprocess

resultado = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--no-header", "-q"],
    cwd=RAIZ, capture_output=True, text=True,
)
print(resultado.stdout[-3000:])
print("Código de salida:", resultado.returncode)
assert resultado.returncode == 0, "Las pruebas automatizadas no pasaron"

## 4. Comprobación de reproducibilidad de extremo a extremo

Se vuelve a ejecutar el pipeline completo desde el consolidado y se comparan los
resultados con el conjunto guardado. Si ambos coinciden, el flujo es
determinista: la misma entrada produce siempre la misma salida.

In [ ]:
df_reejecutado, bitacora_re, reporte_re = pipeline.ejecutar(guardar=False, verboso=False)

comparacion = pd.DataFrame([
    {"indicador": "Registros", "guardado": len(df), "reejecutado": len(df_reejecutado)},
    {"indicador": "Columnas", "guardado": df.shape[1], "reejecutado": df_reejecutado.shape[1]},
    {"indicador": "Suma de sobreduración",
     "guardado": int(df["sobreduracion_sem"].sum()),
     "reejecutado": int(df_reejecutado["sobreduracion_sem"].sum())},
    {"indicador": "Titulación oportuna (n)",
     "guardado": int(df["titulacion_oportuna"].sum()),
     "reejecutado": int(df_reejecutado["titulacion_oportuna"].sum())},
])
comparacion["coincide"] = comparacion["guardado"] == comparacion["reejecutado"]
print(comparacion.to_string(index=False))

assert comparacion["coincide"].all(), "El pipeline no es determinista"
print("\nReproducibilidad verificada: la re-ejecución produce resultados idénticos.")

---

# 5. Resultados

Las secciones siguientes responden, una a una, las preguntas formuladas en la
Fase 1. Todas las tablas se generan con la función `tabla_agregada`, que descarta
los grupos con menos de 30 casos para no publicar promedios calculados sobre
poblaciones mínimas.

## 5.1 Pregunta 1 — ¿Cuánta sobreduración hay y cuántos se titulan a tiempo?

In [ ]:
indicadores = pd.DataFrame([
    {"indicador": "Duración teórica media del plan (semestres)",
     "valor": round(df["dur_total_carr"].mean(), 2)},
    {"indicador": "Duración real media (semestres)",
     "valor": round(df["duracion_real_sem"].mean(), 2)},
    {"indicador": "Sobreduración media (semestres)",
     "valor": round(df["sobreduracion_sem"].mean(), 2)},
    {"indicador": "Sobreduración mediana (semestres)",
     "valor": float(df["sobreduracion_sem"].median())},
    {"indicador": "Índice de duración medio (real/teórica)",
     "valor": round(df["indice_duracion"].mean(), 2)},
    {"indicador": "Titulación oportuna (%)",
     "valor": round(100 * df["titulacion_oportuna"].mean(), 1)},
    {"indicador": "Edad media al titularse (años)",
     "valor": round(df["edad_titulacion"].mean(), 1)},
])
print(indicadores.to_string(index=False))

distribucion = (100 * df["categoria_rezago"].value_counts(normalize=True)).round(1)
print("\nDistribución por categoría de rezago (%):")
print(distribucion.reindex(["Oportuna", "Rezago leve", "Rezago moderado",
                            "Rezago severo"]).to_string())

> **Hallazgo 1.** Solo **19,0 %** de los titulados de pregrado egresa dentro del
> plazo comprometido por su plan de estudios. La sobreduración media es de
> **2,76 semestres** —cerca de un año y medio adicional— y el índice de duración
> medio de **1,43** indica que una carrera toma en promedio un 43 % más de tiempo
> del planificado. Uno de cada cinco titulados (20,4 %) acumula un rezago severo,
> superior a dos años.

## 5.2 Evolución anual y efecto de la pandemia

In [ ]:
por_anio = tr.tabla_agregada(df, "cat_periodo").sort_values("cat_periodo")
print(por_anio.to_string(index=False))

serie = por_anio.set_index("cat_periodo")[["media", "pct_oportuna"]]
fig = viz.serie_temporal(
    serie[["media"]].rename(columns={"media": "Sobreduración media (semestres)"}),
    "Evolución de la sobreduración media en pregrado",
    "Semestres",
    nombre_archivo="f2_3_evolucion_sobreduracion",
)

In [ ]:
fig = viz.serie_temporal(
    serie[["pct_oportuna"]].rename(columns={"pct_oportuna": "Titulación oportuna (%)"}),
    "Evolución del porcentaje de titulación oportuna",
    "Porcentaje de titulados",
    nombre_archivo="f2_3_evolucion_oportuna",
)
print("Variación 2021-2025 en titulación oportuna: "
      f"{serie.loc[2025, 'pct_oportuna'] - serie.loc[2021, 'pct_oportuna']:+.1f} puntos porcentuales")

> **Hallazgo 2.** El año 2020 registra 159.491 titulados de pregrado, un 27,4 %
> menos que 2021: la suspensión de la actividad presencial desplazó exámenes de
> grado y ceremonias hacia el año siguiente. Tras ese ajuste, la titulación
> oportuna mejora de forma sostenida, desde 15,2 % en 2021 hasta 22,3 % en 2025
> (+7,1 puntos porcentuales).

## 5.3 Pregunta 2 — Diferencias por área de conocimiento y tipo de institución

In [ ]:
por_area = tr.tabla_agregada(df, "area_conocimiento")
print(por_area.to_string(index=False))

fig = viz.barras(
    por_area.set_index("area_conocimiento")["media"],
    "Sobreduración media por área de conocimiento",
    "Semestres por sobre la duración teórica",
    nombre_archivo="f2_3_sobreduracion_area",
)

In [ ]:
por_institucion = tr.tabla_agregada(df, "tipo_inst_1")
print(por_institucion.to_string(index=False))

fig = viz.barras(
    por_institucion.set_index("tipo_inst_1")["pct_oportuna"],
    "Titulación oportuna según tipo de institución",
    "Porcentaje de titulados que egresa en el plazo teórico",
    color=config.PALETA["acento"],
    nombre_archivo="f2_3_oportuna_institucion",
)

In [ ]:
cruce = df.pivot_table(
    index="area_conocimiento", columns="tipo_inst_1",
    values="sobreduracion_sem", aggfunc="mean", observed=True,
).round(2)

fig = viz.mapa_calor(
    cruce,
    "Sobreduración media por área y tipo de institución",
    "Semestres",
    nombre_archivo="f2_3_mapa_area_institucion",
)
print(cruce.to_string())

> **Hallazgo 3.** La sobreduración es marcadamente heterogénea: **Derecho**
> (5,77 semestres) y **Ciencias Básicas** (4,50) casi triplican a **Educación**
> (1,93). En Derecho solo el 8,7 % se titula a tiempo, frente al 22,4 % en
> Educación.
>
> Por tipo de institución, las **universidades** promedian 3,32 semestres de
> sobreduración y apenas un 12,8 % de titulación oportuna, mientras que los
> **institutos profesionales** promedian 2,05 y alcanzan un 28,9 %. La brecha se
> explica en parte por la extensión de los planes universitarios y por sus
> procesos de titulación (tesis, exámenes de grado, prácticas prolongadas), que
> en los IP suelen estar incorporados al plan regular.

## 5.4 Pregunta 3 — Brecha de género

In [ ]:
por_genero = tr.tabla_agregada(df, "genero")
print(por_genero.to_string(index=False))

brecha_global = (
    df.loc[df["genero"] == "Mujer", "sobreduracion_sem"].mean()
    - df.loc[df["genero"] == "Hombre", "sobreduracion_sem"].mean()
)
print(f"\nBrecha global (mujeres - hombres): {brecha_global:.2f} semestres")

In [ ]:
brecha_area = tr.razon_feminidad(df, "area_conocimiento")
print(brecha_area.to_string(index=False))

fig = viz.barras_agrupadas(
    brecha_area.set_index("area_conocimiento")[["Hombre", "Mujer"]],
    "Composición por sexo de los titulados según área de conocimiento",
    "Titulados",
    nombre_archivo="f2_3_composicion_genero_area",
)

In [ ]:
medias_genero = df.pivot_table(
    index="area_conocimiento", columns="genero",
    values="sobreduracion_sem", aggfunc="mean", observed=True,
).round(2)

fig = viz.barras_agrupadas(
    medias_genero,
    "Sobreduración media por área y sexo",
    "Semestres por sobre la duración teórica",
    nombre_archivo="f2_3_sobreduracion_genero_area",
)
print(medias_genero.to_string())

> **Hallazgo 4.** Las mujeres se titulan sistemáticamente antes: 2,35 semestres
> de sobreduración frente a 3,29 de los hombres, una brecha de **0,94 semestres**
> a favor de las mujeres, y 21,3 % frente a 16,0 % de titulación oportuna. La
> ventaja femenina se observa en **las diez áreas de conocimiento**, sin
> excepción, y es mayor en Arte y Arquitectura (−0,97) y Educación (−0,88).
>
> La composición por sexo, en cambio, está fuertemente segregada: las mujeres son
> el 80,1 % de los titulados en Educación y el 79,0 % en Salud, pero solo el
> 19,0 % en Tecnología, el área con mayor volumen de titulados del sistema.

## 5.5 Pregunta 4 — Modalidad de estudio y educación a distancia

In [ ]:
por_modalidad = tr.tabla_agregada(df, "modalidad")
print(por_modalidad.to_string(index=False))
print()
print(tr.tabla_agregada(df, "jornada").to_string(index=False))

In [ ]:
evolucion_modalidad = pd.crosstab(
    df["cat_periodo"], df["modalidad"], normalize="index"
).mul(100).round(1)

fig = viz.serie_temporal(
    evolucion_modalidad[["No Presencial"]],
    "Participación de la modalidad no presencial en la titulación de pregrado",
    "Porcentaje de titulados del año",
    nombre_archivo="f2_3_evolucion_no_presencial",
)
print(evolucion_modalidad.to_string())

volumen = pd.crosstab(df["cat_periodo"], df["modalidad"])["No Presencial"]
print(f"\nTitulados no presenciales: {volumen.loc[2020]:,} en 2020 -> "
      f"{volumen.loc[2025]:,} en 2025 "
      f"({100*(volumen.loc[2025]/volumen.loc[2020]-1):.0f}% de aumento)")

In [ ]:
perfil_modalidad = df.groupby("modalidad", observed=True).agg(
    titulados=("edad_titulacion", "size"),
    edad_media=("edad_titulacion", "mean"),
    edad_mediana=("edad_titulacion", "median"),
    sobreduracion_media=("sobreduracion_sem", "mean"),
    pct_oportuna=("titulacion_oportuna", "mean"),
    duracion_teorica_media=("dur_total_carr", "mean"),
).round(2)
perfil_modalidad["pct_oportuna"] = (100 * perfil_modalidad["pct_oportuna"]).round(1)
print(perfil_modalidad.to_string())

> **Hallazgo 5.** La modalidad no presencial pasó de 4,7 % de los titulados de
> pregrado en 2020 a 11,2 % en 2025, triplicando su volumen (de 7.492 a 24.697
> titulados, +230 %). Quienes egresan por esta vía muestran **menos**
> sobreduración (1,84 frente a 2,86 semestres) y casi el doble de titulación
> oportuna (34,6 % frente a 17,2 %).
>
> La comparación, sin embargo, **no debe leerse como un efecto de la modalidad**:
> el perfil es distinto. La edad media de titulación en modalidad no presencial
> es de 37,4 años frente a 27,0 en la presencial, y sus planes son más cortos.
> Se trata en su mayoría de adultos que cursan programas de continuidad, con
> mayor experiencia previa y planes diseñados para trayectorias breves. Aislar el
> efecto propio de la modalidad requiere el control multivariado previsto para la
> Fase 3.

## 5.6 Dimensión territorial

In [ ]:
por_region = tr.tabla_agregada(df, "region_sede")
print(por_region.to_string(index=False))

fig = viz.barras(
    por_region.set_index("region_sede")["media"],
    "Sobreduración media por región de la sede",
    "Semestres por sobre la duración teórica",
    color=config.PALETA["neutro"],
    nombre_archivo="f2_3_sobreduracion_region",
)

> **Hallazgo 6.** La sobreduración es mayor en las regiones extremas del norte y
> del sur —Atacama (3,69), Antofagasta (3,64) y Arica y Parinacota (3,63)— y
> menor en Maule (2,43), O'Higgins (2,54) y la Región Metropolitana (2,58). La
> diferencia entre los extremos supera un semestre completo. La composición de la
> oferta explica parte del patrón: las regiones mineras concentran carreras
> técnicas y de ingeniería, que son precisamente las de mayor rezago.

## 5.7 Carreras con mayor y menor rezago

In [ ]:
por_carrera = tr.tabla_agregada(df, "nomb_carrera", minimo_casos=3_000)

print("Mayor sobreduración (carreras con al menos 3.000 titulados):")
print(por_carrera.head(10).to_string(index=False))
print("\nMenor sobreduración:")
print(por_carrera.tail(10).to_string(index=False))

por_carrera.to_csv(config.TABLES_DIR / "f2_3_sobreduracion_por_carrera.csv", index=False)

In [ ]:
extremos = pd.concat([por_carrera.head(8), por_carrera.tail(8)])
fig = viz.barras(
    extremos.set_index("nomb_carrera")["media"],
    "Carreras con mayor y menor sobreduración (>= 3.000 titulados)",
    "Semestres por sobre la duración teórica",
    nombre_archivo="f2_3_carreras_extremas",
)

## 6. Exportación de tablas de resultados

Todas las tablas que sustentan el informe se guardan en `reports/tables/` para
que las cifras citadas sean verificables y reproducibles.

In [ ]:
tablas = {
    "f2_3_indicadores_generales": indicadores,
    "f2_3_por_anio": por_anio,
    "f2_3_por_area": por_area,
    "f2_3_por_institucion": por_institucion,
    "f2_3_por_genero": por_genero,
    "f2_3_brecha_genero_area": brecha_area,
    "f2_3_por_modalidad": por_modalidad,
    "f2_3_perfil_modalidad": perfil_modalidad.reset_index(),
    "f2_3_por_region": por_region,
}
for nombre, tabla in tablas.items():
    tabla.to_csv(config.TABLES_DIR / f"{nombre}.csv", index=False)

figuras = sorted(p.name for p in config.FIGURES_DIR.glob("*.png"))
print(f"Tablas exportadas: {len(tablas)}")
print(f"Figuras generadas: {len(figuras)}")
for figura in figuras:
    print("  -", figura)

## 7. Síntesis de hallazgos

| N.º | Hallazgo | Evidencia |
|-----|----------|-----------|
| H1 | Solo el 19,0 % de los titulados de pregrado egresa en el plazo teórico; la sobreduración media es de 2,76 semestres (índice 1,43). | Sección 5.1 |
| H2 | La titulación oportuna mejora de 15,2 % (2021) a 22,3 % (2025); 2020 muestra una caída de 27,4 % en el volumen de titulados. | Sección 5.2 |
| H3 | Derecho (5,77) y Ciencias Básicas (4,50) casi triplican la sobreduración de Educación (1,93). | Sección 5.3 |
| H4 | Las universidades promedian 3,32 semestres de rezago frente a 2,05 de los institutos profesionales. | Sección 5.3 |
| H5 | Las mujeres se titulan 0,94 semestres antes que los hombres, sin excepción en las diez áreas. | Sección 5.4 |
| H6 | La modalidad no presencial se triplica entre 2020 y 2025 y presenta menor rezago, pero con un perfil etario 10 años mayor. | Sección 5.5 |
| H7 | Las regiones extremas del norte superan en más de un semestre a las de menor rezago. | Sección 5.6 |

## 8. Limitaciones

1. **Sesgo de selección.** La base contiene únicamente a quienes se titularon.
   Las carreras con mayor deserción podrían mostrar una sobreduración
   artificialmente baja, porque los estudiantes más rezagados abandonan antes de
   titularse.
2. **Cambios de plan no observables.** Si un estudiante cambia de malla, la
   duración teórica registrada puede no ser la que efectivamente cursó.
3. **Registros descartados.** Se excluyó un 4,7 % de los registros de pregrado
   por carecer de año de ingreso utilizable; en su mayoría corresponden a
   estudiantes provenientes de otro programa o institución, cuyo tiempo total de
   estudios no es observable en esta fuente.
4. **Carácter descriptivo.** Las diferencias entre grupos son asociaciones, no
   efectos causales: no se controla por variables simultáneas.

## 9. Proyección a las fases siguientes

| Fase | Trabajo previsto | Insumo que aporta esta entrega |
|------|------------------|--------------------------------|
| F3 | Modelar la sobreduración (regresión) y la titulación oportuna (clasificación), con control multivariado de área, institución, modalidad, sexo y territorio. | `titulados_pregrado_analitico.parquet` con la variable objetivo ya construida y validada |
| F4 | Reporte analítico final y comunicación de hallazgos. | Catálogo de figuras y tablas reproducibles en `reports/` |

> **Fin de la Fase 2.**